In [1]:
import torch
import torch.nn as nn
import torch_pruning as tp
import os
import time

from torchvision.models import mobilenet_v2

In [2]:
# =====================================================
# Load original float model
# =====================================================

pristine_model = mobilenet_v2(weights="DEFAULT")
pristine_model.eval()

original_classifier_weight = (
    pristine_model.classifier[1].weight.detach().clone()
)

original_classifier_bias = (
    pristine_model.classifier[1].bias.detach().clone()
)

print("Original classifier shape:")
print(original_classifier_weight.shape)

Original classifier shape:
torch.Size([1000, 1280])


In [3]:
# =====================================================
# Quantize first
# =====================================================

quantized_model = torch.quantization.quantize_dynamic(
    mobilenet_v2(weights="DEFAULT").eval(),
    {nn.Linear},
    dtype=torch.qint8
)

print(type(quantized_model.classifier[1]))
print(quantized_model.classifier[1])

C:\Users\user\AppData\Local\Temp\ipykernel_27060\2446958789.py:5: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


<class 'torch.ao.nn.quantized.dynamic.modules.linear.Linear'>
DynamicQuantizedLinear(in_features=1280, out_features=1000, dtype=torch.qint8, qscheme=torch.per_tensor_affine)


In [4]:
# =====================================================
# Structural pruning
# =====================================================

example_inputs = torch.randn(1, 3, 224, 224)

importance = tp.importance.MagnitudeImportance(p=2)

ignored_layers = [
    quantized_model.classifier
]

pruner = tp.pruner.MagnitudePruner(
    quantized_model,
    example_inputs,
    importance=importance,
    pruning_ratio=0.2,
    ignored_layers=ignored_layers
)

print("Before pruning:")
print(quantized_model.features[18][0].out_channels)

pruner.step()

print("After pruning:")
print(quantized_model.features[18][0].out_channels)

Before pruning:
1280
After pruning:
1024


In [5]:
# =====================================================
# Recover kept channels
# =====================================================

history = pruner.pruning_history()

removed_idxs = None

for layer_name, is_out_channel, idxs in history:

    if layer_name == "features.18.0":

        removed_idxs = idxs
        break

assert removed_idxs is not None

all_channels = set(range(1280))

kept_idxs = sorted(
    list(all_channels - set(removed_idxs))
)

print("Original channels:", 1280)
print("Removed channels :", len(removed_idxs))
print("Kept channels    :", len(kept_idxs))

Original channels: 1280
Removed channels : 256
Kept channels    : 1024


In [6]:
# =====================================================
# Slice original trained classifier
# =====================================================

kept_idxs_tensor = torch.tensor(
    kept_idxs,
    dtype=torch.long
)

sliced_weight = (
    original_classifier_weight[:, kept_idxs_tensor]
)

sliced_bias = (
    original_classifier_bias.clone()
)

print("Sliced classifier shape:")
print(sliced_weight.shape)

repaired_float_linear = nn.Linear(
    in_features=len(kept_idxs),
    out_features=1000
)

with torch.no_grad():

    repaired_float_linear.weight.copy_(
        sliced_weight
    )

    repaired_float_linear.bias.copy_(
        sliced_bias
    )

Sliced classifier shape:
torch.Size([1000, 1024])


In [7]:
# =====================================================
# Re-quantize the repaired classifier
# =====================================================

repaired_float_linear.qconfig = torch.quantization.default_dynamic_qconfig

repaired_quantized_linear = torch.ao.nn.quantized.dynamic.Linear.from_float(
    repaired_float_linear
)

quantized_model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    repaired_quantized_linear
)

print(quantized_model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): DynamicQuantizedLinear(in_features=1024, out_features=1000, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
)


In [8]:
with torch.no_grad():
    x = torch.randn(1,3,224,224)
    y = quantized_model(x)

print(y.shape)

torch.Size([1, 1000])


In [9]:
total_params = sum(
    p.numel()
    for p in quantized_model.parameters()
)

print(f"Total parameters: {total_params:,}")

Total parameters: 1,435,140


In [10]:
save_path = "../models/Q_then_P_repaired.pth"

torch.save(
    quantized_model.state_dict(),
    save_path
)

size_mb = os.path.getsize(save_path) / (1024 * 1024)

print(f"Model size: {size_mb:.2f} MB")

Model size: 6.66 MB


In [11]:
def benchmark(model, device, input_tensor, runs=100):

    model = model.to(device)

    input_tensor = input_tensor.to(device)

    for _ in range(10):
        with torch.no_grad():
            _ = model(input_tensor)

    if device == "cuda":
        torch.cuda.synchronize()

    start = time.time()

    for _ in range(runs):
        with torch.no_grad():
            _ = model(input_tensor)

    if device == "cuda":
        torch.cuda.synchronize()

    end = time.time()

    return (end - start) / runs


input_tensor = torch.randn(1, 3, 224, 224)

cpu_latency = benchmark(
    quantized_model,
    "cpu",
    input_tensor
)

print(
    f"CPU Average Latency: {cpu_latency:.6f} seconds"
)

CPU Average Latency: 0.121314 seconds
